## Analítica computacional para la toma de decisiones - P1

### Limpieza de datos

In [1]:
import numpy as np
import pandas as pd 

In [2]:
df = pd.read_csv("jbjy-vk9h_F.csv")
print(df.shape)
print(df['nombre_entidad'].unique())

(850, 85)
<StringArray>
['UNGRD']
Length: 1, dtype: str


In [3]:
df = df.map(lambda x: str(x).replace('\n', ' ').replace('\r', ' ') if isinstance(x, str) else x)
df.to_csv("ungrd_limpio.csv", index=False)

In [4]:
df_confirmar = pd.read_csv("ungrd_limpio.csv")
print(df_confirmar.shape)

(850, 85)


In [5]:
df.shape

(850, 85)

#### Cargar el archivo Atenas

In [7]:
df1 = pd.read_csv("P1_bd.csv")
print("Filas y columnas:", df1.shape)
print("\nTipos de dato:")
print(df1.dtypes)
print("\nPrimeras filas:")
df1.head()

Filas y columnas: (850, 7)

Tipos de dato:
nit_entidad                       int64
nombre_entidad                      str
modalidad_de_contratacion           str
valor_del_contrato              float64
fecha_de_firma                      str
fecha_de_inicio_del_contrato        str
fecha_de_fin_del_contrato           str
dtype: object

Primeras filas:


,nit_entidad,nombre_entidad,modalidad_de_contratacion,valor_del_contrato,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato
0,900478966,UNGRD,Mínima cuantía,18476990.0,NaN,NaN,NaN
1,900478966,UNGRD,Contratación directa,28676000.0,2018-01-26T00:00:00.000,2018-01-26T00:00:00.000,2018-05-25T00:00:00.000
2,900478966,UNGRD,Contratación directa,38400000.0,2024-07-29T00:00:00.000,2024-07-30T00:00:00.000,2024-12-23T00:00:00.000
3,900478966,UNGRD,Contratación directa,63390438.0,2023-04-28T00:00:00.000,2023-04-29T00:00:00.000,2023-10-28T00:00:00.000
4,900478966,UNGRD,Contratación directa,20081760.0,2019-01-11T00:00:00.000,2019-01-12T00:00:00.000,2019-06-30T00:00:00.000


### Revisar cada variable

In [8]:
print(df1['nit_entidad'].isnull().sum())
print(df1['nombre_entidad'].isnull().sum())
print(df1['modalidad_de_contratacion'].isnull().sum())
print(df1['valor_del_contrato'].isnull().sum())
print(df1['fecha_de_firma'].isnull().sum())

0
0
0
0
179


#### Revisión y limpieza variable fecha_de_firma

In [9]:
#Se identificó que fecha_de_firma tiene 179 valores faltantes 
#Revisamos si hay alguna tendencia en los faltantes  
df1[df1['fecha_de_firma'].isnull()]['modalidad_de_contratacion'].value_counts()

modalidad_de_contratacion
Contratación directa                           40
Licitación pública                             32
Selección abreviada subasta inversa            31
Selección Abreviada de Menor Cuantía           31
Mínima cuantía                                 22
Concurso de méritos abierto                    18
Licitación pública Obra Publica                 2
Contratación régimen especial (con ofertas)     2
Contratación Directa (con ofertas)              1
Name: count, dtype: int64

In [10]:
#Para contarestar estos faltantes revisamos las demás fehcas que hay disponibles 
print(df1['fecha_de_firma'].isnull().sum())
print(df1['fecha_de_inicio_del_contrato'].isnull().sum())
print(df1['fecha_de_fin_del_contrato'].isnull().sum())

179
229
68


In [11]:
#Creamos fecha_referecia crevisando si existe fecha_de_firma, en caso de que no se utiliza fecha_de_inicio_del_contrato, de lo contrario fecha_de_fin_del_contrato
df1['fecha_referencia'] = (
    df1['fecha_de_firma']
    .fillna(df1['fecha_de_inicio_del_contrato'])
    .fillna(df1['fecha_de_fin_del_contrato'])
)

In [49]:
#Revisamos los faltantes de fecha_referencia
print(df1['fecha_referencia'].isnull().sum())

68


In [50]:
#Convertimos el formato de texto a fecha, luego extrae solo el año
df1['fecha_referencia'] = pd.to_datetime(df1['fecha_referencia'], errors='coerce')
df1['anio'] = df1['fecha_referencia'].dt.year
print(df1[['fecha_referencia', 'anio']].head())

  fecha_referencia    anio
0              NaT     NaN
1       2018-01-26  2018.0
2       2024-07-29  2024.0
3       2023-04-28  2023.0
4       2019-01-11  2019.0


In [51]:
#Se excluyeron las filas que no tengan ninguna de las tres fechas 
df2 = df1.dropna(subset=['anio'])
print(df2.shape)
print(df2[['fecha_referencia', 'anio']].head())

(782, 9)
  fecha_referencia    anio
1       2018-01-26  2018.0
2       2024-07-29  2024.0
3       2023-04-28  2023.0
4       2019-01-11  2019.0
5       2026-01-29  2026.0


In [23]:
columnas_finales = ['nit_entidad', 'nombre_entidad', 'modalidad_de_contratacion', 'valor_del_contrato', 'anio']

df3 = df2[columnas_finales]
print(df3.columns.tolist())
df3.head()

['nit_entidad', 'nombre_entidad', 'modalidad_de_contratacion', 'valor_del_contrato', 'anio']


,nit_entidad,nombre_entidad,modalidad_de_contratacion,valor_del_contrato,anio
1,900478966,UNGRD,Contratación directa,28676000.0,2018.0
2,900478966,UNGRD,Contratación directa,38400000.0,2024.0
3,900478966,UNGRD,Contratación directa,63390438.0,2023.0
4,900478966,UNGRD,Contratación directa,20081760.0,2019.0
5,900478966,UNGRD,Contratación directa,79000000.0,2026.0


#### Revisión y limpieza variable valor_del_contrato

Revisamos los valores = 0 y que % representan en cada categoría, en caso de que no sea significante, los eliminamos

In [24]:
(df3['valor_del_contrato'] == 0).sum()

np.int64(26)

In [25]:
df3[df3['valor_del_contrato'] == 0]['modalidad_de_contratacion'].value_counts()

modalidad_de_contratacion
Contratación directa                    24
Concurso de méritos abierto              1
Selección Abreviada de Menor Cuantía     1
Name: count, dtype: int64

In [26]:
df3['modalidad_de_contratacion'].value_counts()

modalidad_de_contratacion
Contratación directa                           561
Mínima cuantía                                  86
Selección Abreviada de Menor Cuantía            36
Contratación Directa (con ofertas)              31
Selección abreviada subasta inversa             27
Licitación pública                              27
Concurso de méritos abierto                     10
Contratación régimen especial (con ofertas)      3
Licitación pública Obra Publica                  1
Name: count, dtype: int64

In [27]:
df4 = df3[df3['valor_del_contrato'] > 0]

In [28]:
(df4['valor_del_contrato'] == 0).sum()

np.int64(0)

In [29]:
df4['modalidad_de_contratacion'].value_counts()

modalidad_de_contratacion
Contratación directa                           537
Mínima cuantía                                  86
Selección Abreviada de Menor Cuantía            35
Contratación Directa (con ofertas)              31
Selección abreviada subasta inversa             27
Licitación pública                              27
Concurso de méritos abierto                      9
Contratación régimen especial (con ofertas)      3
Licitación pública Obra Publica                  1
Name: count, dtype: int64

Revisamos los valores fuera del rango, en caso de no se significantes, los eliminamos

In [30]:
(df4['valor_del_contrato'] < 10000).sum()

np.int64(3)

In [31]:
df4[df4['valor_del_contrato'] < 10000][['valor_del_contrato', 'modalidad_de_contratacion']]

,valor_del_contrato,modalidad_de_contratacion
189,1.00,Concurso de méritos abierto
366,0.01,Concurso de méritos abierto
501,0.01,Concurso de méritos abierto


In [32]:
df5 = df4[df4['valor_del_contrato'] >= 10000]

In [33]:
(df5['valor_del_contrato'] < 10000).sum()

np.int64(0)

In [34]:
print("Media:", df5['valor_del_contrato'].mean())
print("Mediana:", df5['valor_del_contrato'].median())

Media: 7836558192693.538
Mediana: 38801209.33


In [35]:
print(df5['valor_del_contrato'].max())
print(df5['valor_del_contrato'].sort_values(ascending=False).head(10))

5900683669629647.0
23     5.900684e+15
118    1.896879e+10
156    1.840121e+10
444    1.836845e+10
242    1.794118e+10
303    1.114700e+10
789    9.457383e+09
258    8.044745e+09
92     6.495000e+09
541    6.175328e+09
Name: valor_del_contrato, dtype: float64


In [36]:
df5.loc[23]

nit_entidad                             900478966
nombre_entidad                              UNGRD
modalidad_de_contratacion    Contratación directa
valor_del_contrato             5900683669629647.0
anio                                       2021.0
Name: 23, dtype: object

In [37]:
df6 = df5[df5.index != 23]

In [38]:
print("Media:", df6['valor_del_contrato'].mean())
print("Mediana:", df6['valor_del_contrato'].median())

Media: 325331740.14231384
Mediana: 38746425.165


In [39]:
round(df6['valor_del_contrato'].describe(), 0)

count    7.520000e+02
mean     3.253317e+08
std      1.625456e+09
min      1.302800e+04
25%      1.975000e+07
50%      3.874642e+07
75%      7.900000e+07
max      1.896879e+10
Name: valor_del_contrato, dtype: float64

#### Revisión y limpieza variable modalidad_de_contratacion

In [40]:
df6['modalidad_de_contratacion'].value_counts()

modalidad_de_contratacion
Contratación directa                           536
Mínima cuantía                                  86
Selección Abreviada de Menor Cuantía            35
Contratación Directa (con ofertas)              31
Selección abreviada subasta inversa             27
Licitación pública                              27
Concurso de méritos abierto                      6
Contratación régimen especial (con ofertas)      3
Licitación pública Obra Publica                  1
Name: count, dtype: int64

In [41]:
conteo = df6['modalidad_de_contratacion'].value_counts()

In [42]:
modalidades_grandes = conteo[conteo >= 10].index

In [43]:
df6['modalidad_agrupada'] = df6['modalidad_de_contratacion']

In [44]:
df6.loc[~df6['modalidad_de_contratacion'].isin(modalidades_grandes), 'modalidad_agrupada'] = 'Otras'

In [45]:
df6['modalidad_agrupada'].value_counts()

modalidad_agrupada
Contratación directa                    536
Mínima cuantía                           86
Selección Abreviada de Menor Cuantía     35
Contratación Directa (con ofertas)       31
Selección abreviada subasta inversa      27
Licitación pública                       27
Otras                                    10
Name: count, dtype: int64

### Descargar csv

In [46]:
df6.to_csv("dff_final.csv", index=False)